In [1]:
# the purpose of this code is to calculate the diffusion limited escape flux of total H
# and compute the ocean loss timescale
# the input values are calculated in the Jupyter notebook - MixingRatios 
# also awaiting for e0_branch data

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import matplotlib.ticker as ticker

There are two averaging methods to compute the escape rate:\\

1. Find the global mean, annual mean of mixing ratios and temperature, substitute into the equations
2. Calculae the equations based on the monthly mean, global mean quantities, then take annual average.

In [2]:
# define eccentricity range for controlled experiment

e_range =['circ']

# define path 

data_dir_file = []

for i in range(len(e_range)):
    data_dir_file.append('/nobackup/pybl/cesm_sims/archive/b.e21.BWma1850.f19_g17.new_'+ e_range[i]+'.001/atm/hist/')
    
# define specific path
# we don't want to show variable ending with 'TOA' because it stands for top of atmosphere and it is used for 
# comparingwith observation with an additional atmospheric layer. To show energy budget, please use EEI=FSNT-FLNT.
Path_H2O        = []
Path_H2         = []
Path_H          = []
Path_CH4        = []
Path_OH         = []
Path_T          = []


for i in range(len(data_dir_file)):

    Path_H2O.append(data_dir_file[i]+'H2O_'+ e_range[i] +'_obl0.nc') 
    Path_H2.append(data_dir_file[i]+'H2_'+ e_range[i] +'_obl0.nc')
    Path_H.append(data_dir_file[i]+'H_'+ e_range[i] +'_obl0.nc')
    Path_CH4.append(data_dir_file[i]+'CH4_'+ e_range[i] +'_obl0.nc')
    Path_OH.append(data_dir_file[i]+'OH_'+ e_range[i] +'_obl0.nc')
    Path_T.append(data_dir_file[i]+'T_'+ e_range[i] +'_obl0.nc')


# Import data
# each file has 5 case elements

H2_circ            =[]
H_circ             =[]
CH4_circ           =[]
OH_circ            =[]
H2O_circ           =[]
T_circ             =[]

for i in range(len(e_range)):

    H2O_circ.append(xr.open_mfdataset(Path_H2O[i]))
    H2_circ.append(xr.open_mfdataset(Path_H2[i]))
    H_circ.append(xr.open_mfdataset(Path_H[i]))
    CH4_circ.append(xr.open_mfdataset(Path_CH4[i]))
    OH_circ.append(xr.open_mfdataset(Path_OH[i]))
    T_circ.append(xr.open_mfdataset(Path_T[i]))



In [43]:
# define eccentricity range

e_range =['e4']

# define path 

data_dir_file = []

for i in range(len(e_range)):
    data_dir_file.append('/nobackup/pybl/cesm_sims/archive/b.e21.BWma1850.f19_g17.new_'+ e_range[i]+'_test.001/atm/hist/')
    
# define specific path
# we don't want to show variable ending with 'TOA' because it stands for top of atmosphere and it is used for 
# comparingwith observation with an additional atmospheric layer. To show energy budget, please use EEI=FSNT-FLNT.

Path_H2O        = []
Path_H2         = []
Path_H          = []
Path_CH4        = []
Path_T          = []


for i in range(len(data_dir_file)):

    Path_H2O.append(data_dir_file[i]+'H2O_'+ e_range[i] +'_merge.nc') 
    Path_H2.append(data_dir_file[i]+'H2_'+ e_range[i] +'_merge.nc')
    Path_H.append(data_dir_file[i]+'H_'+ e_range[i] +'_merge.nc')
    Path_CH4.append(data_dir_file[i]+'CH4_'+ e_range[i] +'_merge.nc')
    Path_T.append(data_dir_file[i]+'T_'+ e_range[i] +'_merge.nc')

    


# Import data
# each file has 5 case elements

H2_e4            =[]
H_e4             =[]
CH4_e4           =[]
H2O_e4           =[]
T_e4             =[]


for i in range(len(e_range)):

    H2O_e4.append(xr.open_mfdataset(Path_H2O[i]))
    H2_e4.append(xr.open_mfdataset(Path_H2[i]))
    H_e4.append(xr.open_mfdataset(Path_H[i]))
    CH4_e4.append(xr.open_mfdataset(Path_CH4[i]))
    T_e4.append(xr.open_mfdataset(Path_T[i]))




In [10]:
def H_flux(T, T_homo, H_homo, H2_homo,H2O_homo, CH4_homo):

# thermospheric temperature at model top
# I would intend to take global mean for this value 

    b_H        = (6.5*10**17)*T_homo**0.7 
    
# for single H atom, A=4.8*10**17
# if consider more, such as H2, needs to use weighted b12=(b_h*n_h+b_h2*n_h2)/(n_h+n_h2)

    b_H2       = (2.67*10**17)*T_homo**0.75 

# https://journals.ametsoc.org/view/journals/atsc/30/8/1520-0469_1973_030_1481_teolgf_2_0_co_2.xml?tab_body=pdf
# The Escape of Light Gases from Planetary Atmospheres
# Donald M. Hunten
#     n_h1       = H_thermo
    
#     n_h2       = H2_thermo

    b_H2O       = (0.137*10**17)*T_homo**1.072
    
    b_CH4       = (0.756*10**17)*T_homo**0.747
    
#     b_weighted=(b_H*n_h1+b_H2*n_h2)/(n_h1+n_h2)
    
#     print(b_weighted.mean(dim='time').values)
    
#     f_tot     =  H_thermo + 2*H2_thermo
    
# source: 
#Catling, David C.; Kasting, James F. (2017). 
#Atmospheric Evolution on Inhabited and Lifeless Worlds. 
#Cambridge: Cambridge University Press. 
#doi:10.1017/9781139020558. ISBN 9781139020558.

# thermospheric hydrogen mixing ratio (model top)
# I would intend to take global mean for this value 

    K        = 1.38 * 10**(-23) # J·K−1

    T_air    = T_homo #  scale height depends on temperature, here we evaluate everything in homopause

    u        = 1.660*10**(-27) # hydrogen mass in kg

    m        = 28.964 * u # mean atmospheric molecular mass in kg

    g        = 9.8 # m/s2

    H        = (K*T_air/(m*g))*100  # in cm
    

    Phi_H =(b_H*H_homo/H) + (b_H2*2*H2_homo/H) +(b_H2O*2*H2O_homo/H)+(b_CH4*4*CH4_homo/H)  #b_weighted *f_tot/H   
    
#     print((b_H/H).values)
    
#     print(H_thermo.values)

    return Phi_H 

# goal: plot it as a function of time for each eccentricity

In [9]:
def weighted_temporal_mean(ds, var):
    """
    weight by days in each month
    """
    # Determine the month length
    month_length = ds.time.dt.days_in_month

    # Calculate the weights
    wgts = month_length.groupby("time.year") / month_length.groupby("time.year").sum()

    # Make sure the weights in each year add up to 1
    np.testing.assert_allclose(wgts.groupby("time.year").sum(xr.ALL_DIMS), 1.0)

    # Subset our dataset for our variable
    obs = ds[var]

    # Setup our masking for nan values
    cond = obs.isnull()
    ones = xr.where(cond, 0.0, 1.0)

    # Calculate the numerator
    obs_sum = (obs * wgts).resample(time="AS").sum(dim="time")

    # Calculate the denominator
    ones_out = (ones * wgts).resample(time="AS").sum(dim="time")

    # Return the weighted average
    return obs_sum / ones_out

In [20]:
# circ

weights=np.cos(np.deg2rad(T_circ[0].T.lat))

T_circ_global_annual=weighted_temporal_mean(T_circ[0].sel(time=slice('0012-01', '0016-12')),"T").weighted(weights).mean(dim=("lat","lon")).mean(dim='time')

T_circ_hp_global_annual=T_circ_global_annual.isel(lev=11)

H_circ_hp  =weighted_temporal_mean(H_circ[0].sel(time=slice('0012-01', '0016-12')),"H").weighted(weights).mean(dim=("lat","lon")).mean(dim='time').isel(lev=11)
H2_circ_hp =weighted_temporal_mean(H2_circ[0].sel(time=slice('0012-01', '0016-12')),"H2").weighted(weights).mean(dim=("lat","lon")).mean(dim='time').isel(lev=11)
H2O_circ_hp=weighted_temporal_mean(H2O_circ[0].sel(time=slice('0012-01', '0016-12')),"H2O").weighted(weights).mean(dim=("lat","lon")).mean(dim='time').isel(lev=11)
CH4_circ_hp=weighted_temporal_mean(CH4_circ[0].sel(time=slice('0012-01', '0016-12')),"CH4").weighted(weights).mean(dim=("lat","lon")).mean(dim='time').isel(lev=11)



In [44]:
# e4

T_e4_global_annual=weighted_temporal_mean(T_e4[0].sel(time=slice('0012-01', '0016-12')),"T").weighted(weights).mean(dim=("lat","lon")).mean(dim='time')

T_e4_hp_global_annual=T_e4_global_annual.isel(lev=11)

H_e4_hp  =weighted_temporal_mean(H_e4[0].sel(time=slice('0012-01', '0016-12')),"H").weighted(weights).mean(dim=("lat","lon")).mean(dim='time').isel(lev=11)
H2_e4_hp =weighted_temporal_mean(H2_e4[0].sel(time=slice('0012-01', '0016-12')),"H2").weighted(weights).mean(dim=("lat","lon")).mean(dim='time').isel(lev=11)
H2O_e4_hp=weighted_temporal_mean(H2O_e4[0].sel(time=slice('0012-01', '0016-12')),"H2O").weighted(weights).mean(dim=("lat","lon")).mean(dim='time').isel(lev=11)
CH4_e4_hp=weighted_temporal_mean(CH4_e4[0].sel(time=slice('0012-01', '0016-12')),"CH4").weighted(weights).mean(dim=("lat","lon")).mean(dim='time').isel(lev=11)



In [32]:
from decimal import Decimal

H_escape_circ =H_flux(T_circ_global_annual, T_circ_hp_global_annual, H_circ_hp, H2_circ_hp,H2O_circ_hp, CH4_circ_hp).values


# number of atoms/cm^2/s

print('H escape rate for e=0.4 in #/cm^2',"{:e}".format(H_escape_circ))

water_content = 1.4 * 10 **24 # g 

H_content     = 2* ((1.4 * 10 **24 /18.02)*(6.022*10**23))   # number of H atoms in ocean water

print('H_content', H_content)

# # water_content source: 
# # 1. WOLF AND TOON 2015: The evolution of habitable climates under the brightening Sun
# # 2. Kopparapu 2017 HABITABLE MOIST ATMOSPHERES ON TERRESTRIAL PLANETS NEAR THE INNER EDGE 
# # OF THE HABITABLE ZONE AROUND M-DWARFS


Z_altitude = 10000000 # in cm

diffusion_area = 4 * np.pi * (637100720+Z_altitude)**2 # cm^2

print('diffusion_area in cm2',diffusion_area)

total_escape_circ="{:e}".format(diffusion_area*H_escape_circ)

print('total escape rate #atoms per sec for e=0.4',total_escape_circ)


tau_circ    = H_content/((H_escape_circ*diffusion_area)*(3600*24*365))

print('tau_circ in Gyr',tau_circ/10**9)


H escape rate for e=0.4 in #/cm^2 2.286696e+08
H_content 9.35715871254162e+46
diffusion_area in cm2 5.262033760179728e+18
total escape rate #atoms per sec for e=0.4 1.203267e+27
tau_circ in Gyr 2465.899500238361


In [45]:
from decimal import Decimal

H_escape_e4 =H_flux(T_e4_global_annual, T_e4_hp_global_annual, H_e4_hp, H2_e4_hp,H2O_e4_hp, CH4_e4_hp).values
# number of atoms/cm^2/s

print('H escape rate for e=0.4 in #/cm^2',"{:e}".format(H_escape_e4))

water_content = 1.4 * 10 **24 # g 

H_content     = 2* ((1.4 * 10 **24 /18.02)*(6.022*10**23))   # number of H atoms in ocean water

print('H_content', H_content)

# # water_content source: 
# # 1. WOLF AND TOON 2015: The evolution of habitable climates under the brightening Sun
# # 2. Kopparapu 2017 HABITABLE MOIST ATMOSPHERES ON TERRESTRIAL PLANETS NEAR THE INNER EDGE 
# # OF THE HABITABLE ZONE AROUND M-DWARFS


Z_altitude = 10000000 # in cm

diffusion_area = 4 * np.pi * (637100720+Z_altitude)**2 # cm^2

print('diffusion_area in cm2',diffusion_area)

total_escape_e4="{:e}".format(diffusion_area*H_escape_e4)

print('total escape rate #atoms per sec for e=0.4',total_escape_e4)


tau_e4    = H_content/((H_escape_e4*diffusion_area)*(3600*24*365))

print('tau_e4 in Gyr',tau_e4/10**9)


H escape rate for e=0.4 in #/cm^2 6.411590e+08
H_content 9.35715871254162e+46
diffusion_area in cm2 5.262033760179728e+18
total escape rate #atoms per sec for e=0.4 3.373800e+27
tau_e4 in Gyr 879.4638851208816


Now, second average method

In [51]:
# circ

T_circ_global_monthly=T_circ[0].T.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time')

T_circ_hp_global_monthly=T_circ_global_annual.isel(lev=11)

H_circ_hp   =  H_circ[0].H.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time').isel(lev=11)
H2_circ_hp  = H2_circ[0].H2.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time').isel(lev=11)
H2O_circ_hp =H2O_circ[0].H2O.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time').isel(lev=11)
CH4_circ_hp =CH4_circ[0].CH4.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time').isel(lev=11)



In [161]:
# e4

T_e4_global_monthly=T_e4[0].T.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time')

T_e4_hp_global_monthly=T_e4_global_annual.isel(lev=11)

H_e4_hp   =  H_e4[0].H.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time').isel(lev=11)
H2_e4_hp  = H2_e4[0].H2.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time').isel(lev=11)
H2O_e4_hp =H2O_e4[0].H2O.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time').isel(lev=11)
CH4_e4_hp =CH4_e4[0].CH4.sel(time=slice('0012-01', '0016-12')).weighted(weights).mean(dim=("lat","lon")).groupby('time.month').mean(dim='time').isel(lev=11)


In [155]:
def weighted_annual_mean(ds):
    
    month_length=T_circ[0].T.sel(time=slice('0012-01', '0012-12')).time.dt.days_in_month
    
    month_length=month_length.groupby("time.month").sum()
    
    weights_month = month_length/365
    
    np.testing.assert_allclose(weights_month.sum().values, np.ones(1))
    
    ds_weighted = (ds["var"]*weights_month).sum()
    
    return ds_weighted

In [159]:
from decimal import Decimal

H_escape_circ =H_flux(T_circ_global_monthly, T_circ_hp_global_monthly, H_circ_hp, H2_circ_hp,H2O_circ_hp, CH4_circ_hp)#.values

H_escape_circ['var'] = (['month'], H_escape_circ.values)

# number of atoms/cm^2/s

print('H escape rate for e=0.4 in #/cm^2',"{:e}".format(weighted_annual_mean(H_escape_circ).values))

water_content = 1.4 * 10 **24 # g 

H_content     = 2* ((1.4 * 10 **24 /18.02)*(6.022*10**23))   # number of H atoms in ocean water

print('H_content', H_content)

# # # water_content source: 
# # # 1. WOLF AND TOON 2015: The evolution of habitable climates under the brightening Sun
# # # 2. Kopparapu 2017 HABITABLE MOIST ATMOSPHERES ON TERRESTRIAL PLANETS NEAR THE INNER EDGE 
# # # OF THE HABITABLE ZONE AROUND M-DWARFS


Z_altitude = 10000000 # in cm

diffusion_area = 4 * np.pi * (637100720+Z_altitude)**2 # cm^2

print('diffusion_area in cm2',diffusion_area)

total_escape_circ="{:e}".format(diffusion_area*weighted_annual_mean(H_escape_circ).values)

print('total escape rate #atoms per sec for e=0.4',total_escape_circ)

tau_circ    = H_content/((weighted_annual_mean(H_escape_circ).values*diffusion_area)*(3600*24*365))

print('tau_circ in Gyr',tau_circ/10**9)


H escape rate for e=0.4 in #/cm^2 2.286696e+08
H_content 9.35715871254162e+46
diffusion_area in cm2 5.262033760179728e+18
total escape rate #atoms per sec for e=0.4 1.203267e+27
tau_circ in Gyr 2465.8995002383613


In [162]:
from decimal import Decimal

H_escape_e4 =H_flux(T_e4_global_monthly, T_e4_hp_global_monthly, H_e4_hp, H2_e4_hp,H2O_e4_hp, CH4_e4_hp)#.values

H_escape_e4['var'] = (['month'], H_escape_e4.values)

# number of atoms/cm^2/s

print('H escape rate for e=0.4 in #/cm^2',"{:e}".format(weighted_annual_mean(H_escape_e4).values))

water_content = 1.4 * 10 **24 # g 

H_content     = 2* ((1.4 * 10 **24 /18.02)*(6.022*10**23))   # number of H atoms in ocean water

print('H_content', H_content)

# # # water_content source: 
# # # 1. WOLF AND TOON 2015: The evolution of habitable climates under the brightening Sun
# # # 2. Kopparapu 2017 HABITABLE MOIST ATMOSPHERES ON TERRESTRIAL PLANETS NEAR THE INNER EDGE 
# # # OF THE HABITABLE ZONE AROUND M-DWARFS


Z_altitude = 10000000 # in cm

diffusion_area = 4 * np.pi * (637100720+Z_altitude)**2 # cm^2

print('diffusion_area in cm2',diffusion_area)

total_escape_e4="{:e}".format(diffusion_area*weighted_annual_mean(H_escape_e4).values)

print('total escape rate #atoms per sec for e=0.4',total_escape_e4)

tau_e4    = H_content/((weighted_annual_mean(H_escape_e4).values*diffusion_area)*(3600*24*365))

print('tau_e4 in Gyr',tau_e4/10**9)


H escape rate for e=0.4 in #/cm^2 6.441879e+08
H_content 9.35715871254162e+46
diffusion_area in cm2 5.262033760179728e+18
total escape rate #atoms per sec for e=0.4 3.389738e+27
tau_e4 in Gyr 875.32875647303


In [ ]:
def H_flux_alone(T, T_thermo, H_thermo):

# thermospheric temperature at model top
# I would intend to take global mean for this value 

    #b_H        = (4.8*10**17)*T_thermo**0.7 
    
# for single H atom, A=4.8*10**17
# if consider more, such as H2, needs to use weighted b12=(b_h*n_h+b_h2*n_h2)/(n_h+n_h2)

    b_H2       = (2.7*10**17)*T_thermo**0.7 

#     n_h1       = H_thermo
    
#     n_h2       = H2_thermo

    
#     b_weighted=(b_H*n_h1+b_H2*n_h2)/(n_h1+n_h2)
    
#     print(b_weighted.mean(dim='time').values)
    
#     f_tot     =  H_thermo + 2*H2_thermo
    
# source: 
#Catling, David C.; Kasting, James F. (2017). 
#Atmospheric Evolution on Inhabited and Lifeless Worlds. 
#Cambridge: Cambridge University Press. 
#doi:10.1017/9781139020558. ISBN 9781139020558.

# thermospheric hydrogen mixing ratio (model top)
# I would intend to take global mean for this value 

    K        = 1.38 * 10**(-23) # J·K−1

    T_air    = T.mean(dim='lev') # # atmospheric mean temperature in K

    u        = 1.660*10**(-27) # hydrogen mass in kg

    m        = 28.964 * u # mean atmospheric molecular mass in kg

    g        = 9.8 # m/s2

    H        = (K*T_air/(m*g))*100  # in cm
    

    Phi_H =(b_H2*H_thermo)/H #+ (b_H2*2*H2_thermo)/H  #b_weighted *f_tot /H 

    return Phi_H 

# goal: plot it as a function of time for each eccentricity

In [ ]:
# def scale_height(DS):
#     K        = 1.38 * 10**(-23)       # J·K−1
#     T_air    = DS.mean(dim='lev')     # atmospheric mean temperature in K
    
#     print('T_air',T_air.values)
    
#     u        = 1.660*10**(-27)        # hydrogen mass in kg
#     m        = 28.964 * u             # mean atmospheric molecular mass in kg
#     g        = 9.8                    # m/s2
#     H        = (K*T_air/(m*g))/1000     # in km

#     return H
# def Alt_conversion(DS):
    
#     H=scale_height(DS)
    
#     Z=-np.log(DS.lev/DS.lev[-1])*H
    
#     return Z